In [1]:
!pip install -q onnx onnxscript onnxruntime-gpu tensorrt
!nvidia-smi

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
Thu Nov 26 03:15:32 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|==========

In [2]:
import os, time, json
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

SEED = 42
torch.manual_seed(SEED); np.random.seed(SEED)
assert torch.cuda.is_available(), "Switch the Colab runtime to a GPU - TensorRT needs one."
device = torch.device("cuda")
os.makedirs("artefacts", exist_ok=True)
print("PyTorch:", torch.__version__, "| GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.14.1+cu130 | GPU: Tesla T4


In [3]:
class SimpleCNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 16, 3, padding=1)
        self.conv2 = nn.Conv2d(16, 32, 3, padding=1)
        self.pool  = nn.MaxPool2d(2)
        self.fc1   = nn.Linear(32 * 7 * 7, 128)
        self.fc2   = nn.Linear(128, num_classes)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))   # 28x28 -> 14x14
        x = self.pool(F.relu(self.conv2(x)))   # 14x14 -> 7x7
        x = torch.flatten(x, 1)
        x = F.relu(self.fc1(x))
        return self.fc2(x)

model = SimpleCNN().to(device)
n_params = sum(p.numel() for p in model.parameters())
print(model)
print(f"Trainable parameters: {n_params:,}")

SimpleCNN(
  (conv1): Conv2d(1, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv2): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (fc1): Linear(in_features=1568, out_features=128, bias=True)
  (fc2): Linear(in_features=128, out_features=10, bias=True)
)
Trainable parameters: 206,922


In [4]:
tfm = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.1307,), (0.3081,))])
train_ds = datasets.MNIST("data", train=True,  download=True, transform=tfm)
test_ds  = datasets.MNIST("data", train=False, download=True, transform=tfm)
train_loader = DataLoader(train_ds, batch_size=128, shuffle=True)
test_loader  = DataLoader(test_ds,  batch_size=256, shuffle=False)
print(len(train_ds), "train /", len(test_ds), "test images")

100%|██████████| 9.91M/9.91M [00:00<00:00, 15.2MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 397kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 3.75MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 11.2MB/s]

60000 train / 10000 test images


In [5]:
def evaluate_torch(m):
    m.eval(); correct = 0
    with torch.no_grad():
        for x, y in test_loader:
            correct += (m(x.to(device)).argmax(1).cpu() == y).sum().item()
    return correct / len(test_ds)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
EPOCHS = 3
for epoch in range(1, EPOCHS + 1):
    model.train(); running = 0.0
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        loss = F.cross_entropy(model(x), y)
        loss.backward(); optimizer.step()
        running += loss.item()
    print(f"epoch {epoch}: loss={running/len(train_loader):.4f}  test_acc={evaluate_torch(model):.4f}")
torch_acc = evaluate_torch(model)

epoch 1: loss=0.2217  test_acc=0.9770
epoch 2: loss=0.0597  test_acc=0.9868
epoch 3: loss=0.0425  test_acc=0.9883


In [6]:
CKPT = "artefacts/mnist_cnn.pt"
torch.save({
    "model_state_dict": model.state_dict(),
    "optimizer_state_dict": optimizer.state_dict(),
    "epoch": EPOCHS,
    "test_accuracy": torch_acc,
    "config": {"num_classes": 10, "input_shape": [1, 28, 28]},
}, CKPT)

ckpt = torch.load(CKPT, map_location="cpu")
print("Checkpoint keys:", list(ckpt.keys()))
for name, t in ckpt["model_state_dict"].items():
    print(f"{name:14s} {str(tuple(t.shape)):18s} {t.dtype}")
print(f"File size: {os.path.getsize(CKPT)/1024:.1f} KB")

Checkpoint keys: ['model_state_dict', 'optimizer_state_dict', 'epoch', 'test_accuracy', 'config']
conv1.weight   (16, 1, 3, 3)      torch.float32
conv1.bias     (16,)              torch.float32
conv2.weight   (32, 16, 3, 3)     torch.float32
conv2.bias     (32,)              torch.float32
fc1.weight     (128, 1568)        torch.float32
fc1.bias       (128,)             torch.float32
fc2.weight     (10, 128)          torch.float32
fc2.bias       (10,)              torch.float32
File size: 2435.0 KB


In [7]:
restored = SimpleCNN().to(device)
restored.load_state_dict(torch.load(CKPT, map_location=device)["model_state_dict"])
restored.eval()

x = torch.randn(8, 1, 28, 28, device=device)
model.eval()
with torch.no_grad():
    diff = (model(x) - restored(x)).abs().max().item()
print("Max |original - restored| =", diff)
assert diff == 0.0

Max |original - restored| = 0.0


In [8]:
ONNX_PATH = "artefacts/mnist_cnn.onnx"
dummy = torch.randn(1, 1, 28, 28, device=device)
restored.eval()

kwargs = dict(input_names=["input"], output_names=["logits"],
              dynamic_axes={"input": {0: "batch"}, "logits": {0: "batch"}},
              opset_version=17)
try:
    torch.onnx.export(restored, dummy, ONNX_PATH, dynamo=False, **kwargs)  # classic TorchScript exporter
except TypeError:
    torch.onnx.export(restored, dummy, ONNX_PATH, **kwargs)                # older PyTorch versions
print(f"Saved {ONNX_PATH}  ({os.path.getsize(ONNX_PATH)/1024:.1f} KB)")

Saved artefacts/mnist_cnn.onnx  (809.7 KB)


/tmp/ipykernel_3051/952851816.py:9: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(restored, dummy, ONNX_PATH, dynamo=False, **kwargs)  # classic TorchScript exporter


In [9]:
import onnx
from collections import Counter

onnx_model = onnx.load(ONNX_PATH)
onnx.checker.check_model(onnx_model)
print("ONNX checker: model is valid")
print("IR version:", onnx_model.ir_version, "| opset:", onnx_model.opset_import[0].version)

def dims(t):
    return [d.dim_param or d.dim_value for d in t.type.tensor_type.shape.dim]

print("Inputs :", [(i.name, dims(i)) for i in onnx_model.graph.input if i.name == "input"])
print("Outputs:", [(o.name, dims(o)) for o in onnx_model.graph.output])
print("\nOperator counts:", dict(Counter(n.op_type for n in onnx_model.graph.node)))
print("Initializers (weights):", len(onnx_model.graph.initializer))

print("\nNode sequence:")
for i, n in enumerate(onnx_model.graph.node):
    print(f"{i:2d}  {n.op_type:8s} {list(n.input)} -> {list(n.output)}")

ONNX checker: model is valid
IR version: 8 | opset: 17
Inputs : [('input', ['batch', 1, 28, 28])]
Outputs: [('logits', ['batch', 10])]

Operator counts: {'Conv': 2, 'Relu': 3, 'MaxPool': 2, 'Flatten': 1, 'Gemm': 2}
Initializers (weights): 8

Node sequence:
 0  Conv     ['input', 'conv1.weight', 'conv1.bias'] -> ['/conv1/Conv_output_0']
 1  Relu     ['/conv1/Conv_output_0'] -> ['/Relu_output_0']
 2  MaxPool  ['/Relu_output_0'] -> ['/pool/MaxPool_output_0']
 3  Conv     ['/pool/MaxPool_output_0', 'conv2.weight', 'conv2.bias'] -> ['/conv2/Conv_output_0']
 4  Relu     ['/conv2/Conv_output_0'] -> ['/Relu_1_output_0']
 5  MaxPool  ['/Relu_1_output_0'] -> ['/pool_1/MaxPool_output_0']
 6  Flatten  ['/pool_1/MaxPool_output_0'] -> ['/Flatten_output_0']
 7  Gemm     ['/Flatten_output_0', 'fc1.weight', 'fc1.bias'] -> ['/fc1/Gemm_output_0']
 8  Relu     ['/fc1/Gemm_output_0'] -> ['/Relu_2_output_0']
 9  Gemm     ['/Relu_2_output_0', 'fc2.weight', 'fc2.bias'] -> ['logits']


In [10]:
import onnxruntime as ort
print("Available providers:", ort.get_available_providers())

sess = ort.InferenceSession(ONNX_PATH, providers=["CUDAExecutionProvider", "CPUExecutionProvider"])
x_np = np.random.randn(16, 1, 28, 28).astype(np.float32)   # note: batch 16, graph was exported with 1
ort_out = sess.run(["logits"], {"input": x_np})[0]
with torch.no_grad():
    pt_out = restored(torch.from_numpy(x_np).to(device)).cpu().numpy()

print("Max abs diff (PyTorch vs ONNX Runtime):", np.abs(pt_out - ort_out).max())
np.testing.assert_allclose(pt_out, ort_out, rtol=1e-3, atol=1e-4)
print("Dynamic batch works and outputs match.")

Available providers: ['TensorrtExecutionProvider', 'CUDAExecutionProvider', 'CPUExecutionProvider']
Max abs diff (PyTorch vs ONNX Runtime): 1.1920929e-06
Dynamic batch works and outputs match.


In [11]:

import tensorrt as trt
print("TensorRT:", trt.__version__)

ENGINE_PATH = "artefacts/mnist_cnn.engine"

logger = trt.Logger(trt.Logger.WARNING)
builder = trt.Builder(logger)

network = builder.create_network(0)
parser = trt.OnnxParser(network, logger)

with open(ONNX_PATH, "rb") as f:
    if not parser.parse(f.read()):
        for i in range(parser.num_errors):
            print(parser.get_error(i))
        raise RuntimeError("ONNX parsing failed")

config = builder.create_builder_config()

config.set_memory_pool_limit(
    trt.MemoryPoolType.WORKSPACE, 1 << 30
)

print("Building TensorRT engine in FP32...")

# Optimization profile for dynamic batch size
profile = builder.create_optimization_profile()

profile.set_shape(
    "input",
    (1, 1, 28, 28),
    (32, 1, 28, 28),
    (256, 1, 28, 28)
)

config.add_optimization_profile(profile)

import time

t0 = time.time()

plan = builder.build_serialized_network(network, config)

assert plan is not None, "Engine build failed"

with open(ENGINE_PATH, "wb") as f:
    f.write(plan)

print(
    f"Engine built in {time.time()-t0:.1f}s "
    f"-> {ENGINE_PATH} "
    f"({os.path.getsize(ENGINE_PATH)/1024:.1f} KB)"
)


TensorRT: 11.5.0.63
Building TensorRT engine in FP32...
Engine built in 2.7s -> artefacts/mnist_cnn.engine (896.8 KB)


In [12]:
runtime = trt.Runtime(logger)
with open(ENGINE_PATH, "rb") as f:
    engine = runtime.deserialize_cuda_engine(f.read())
context = engine.create_execution_context()

for i in range(engine.num_io_tensors):
    name = engine.get_tensor_name(i)
    print(f"{engine.get_tensor_mode(name).name:7s} {name:8s} dtype={engine.get_tensor_dtype(name)}  shape={engine.get_tensor_shape(name)}")
print("Layers/kernels are hidden inside the plan; the engine is a black box compared with ONNX.")

INPUT   input    dtype=DataType.FLOAT  shape=(-1, 1, 28, 28)
OUTPUT  logits   dtype=DataType.FLOAT  shape=(-1, 10)
Layers/kernels are hidden inside the plan; the engine is a black box compared with ONNX.


In [13]:
def trt_infer(batch: torch.Tensor) -> torch.Tensor:
    batch = batch.contiguous().float().to(device)
    out = torch.empty((batch.shape[0], 10), dtype=torch.float32, device=device)
    context.set_input_shape("input", tuple(batch.shape))
    context.set_tensor_address("input",  batch.data_ptr())
    context.set_tensor_address("logits", out.data_ptr())
    context.execute_async_v3(torch.cuda.current_stream().cuda_stream)
    torch.cuda.current_stream().synchronize()
    return out

x16 = torch.from_numpy(x_np)
trt_out = trt_infer(x16).cpu().numpy()
print("Max abs diff (PyTorch vs TensorRT):", np.abs(pt_out - trt_out).max())
print("Same predicted classes:", (pt_out.argmax(1) == trt_out.argmax(1)).all())

Max abs diff (PyTorch vs TensorRT): 9.536743e-07
Same predicted classes: True


In [14]:
def evaluate_fn(predict):
    correct = 0
    for x, y in test_loader:
        correct += (predict(x).argmax(1) == y.numpy()).sum()
    return correct / len(test_ds)

acc_ckpt = evaluate_fn(lambda x: restored(x.to(device)).detach().cpu().numpy())
acc_onnx = evaluate_fn(lambda x: sess.run(["logits"], {"input": x.numpy()})[0])
acc_trt  = evaluate_fn(lambda x: trt_infer(x).cpu().numpy())

print(f"PyTorch (trained model) : {torch_acc:.4f}")
print(f"Checkpoint reloaded     : {acc_ckpt:.4f}")
print(f"ONNX Runtime            : {acc_onnx:.4f}")
print(f"TensorRT engine (FP16)  : {acc_trt:.4f}")

PyTorch (trained model) : 0.9883
Checkpoint reloaded     : 0.9883
ONNX Runtime            : 0.9883
TensorRT engine (FP16)  : 0.9883


In [15]:
def bench(fn, batch, warmup=20, iters=200):
    for _ in range(warmup): fn(batch)
    torch.cuda.synchronize(); t0 = time.perf_counter()
    for _ in range(iters): fn(batch)
    torch.cuda.synchronize()
    return (time.perf_counter() - t0) / iters * 1000   # ms per batch

B = 64
xb = torch.randn(B, 1, 28, 28, device=device)
xb_np = xb.cpu().numpy()

def pt_fn(b):
    with torch.no_grad(): return restored(b)

results = {
    "PyTorch":      bench(pt_fn, xb),
    "ONNX Runtime": bench(lambda b: sess.run(["logits"], {"input": xb_np}), xb),
    "TensorRT":     bench(trt_infer, xb),
}
for k, v in results.items():
    print(f"{k:13s} {v:7.3f} ms / batch of {B}")
print("\nNote: ONNX Runtime timing includes host<->device copies, so compare trends, not exact ratios.")

PyTorch         0.341 ms / batch of 64
ONNX Runtime    0.479 ms / batch of 64
TensorRT        0.273 ms / batch of 64

Note: ONNX Runtime timing includes host<->device copies, so compare trends, not exact ratios.


In [16]:
import pandas as pd
rows = [
    ("Architecture (code)",   "SimpleCNN class",  "-",                                   "Structure only, no weights"),
    ("Trained checkpoint",    CKPT,               os.path.getsize(CKPT)/1024,            "Weights + optimizer + metadata (needs the class to reload)"),
    ("ONNX graph",            ONNX_PATH,          os.path.getsize(ONNX_PATH)/1024,       "Portable graph + weights, framework-neutral"),
    ("TensorRT engine",       ENGINE_PATH,        os.path.getsize(ENGINE_PATH)/1024,     "Optimised for this GPU + TensorRT version"),
]
df = pd.DataFrame(rows, columns=["Stage", "File", "Size (KB)", "What it holds"])
display(df)

,Stage,File,Size (KB),What it holds
0,Architecture (code),SimpleCNN class,-,"Structure only, no weights"
1,Trained checkpoint,artefacts/mnist_cnn.pt,2434.952148,Weights + optimizer + metadata (needs the clas...
2,ONNX graph,artefacts/mnist_cnn.onnx,809.740234,"Portable graph + weights, framework-neutral"
3,TensorRT engine,artefacts/mnist_cnn.engine,896.839844,Optimised for this GPU + TensorRT version
